# บทเรียนที่ 04 - รูปแบบการออกแบบการใช้เครื่องมือ

ในบทเรียนนี้ คุณจะได้เรียนรู้รูปแบบการออกแบบ **การใช้เครื่องมือ** สำหรับเอเย่นต์ AI โดยใช้ Microsoft Agent Framework (Python) โดยเราจะครอบคลุม:

- การกำหนดฟังก์ชันเครื่องมือด้วยตัวตกแต่ง `@tool` และพารามิเตอร์ที่มีการกำหนดชนิด
- การจัดเตรียมสคีมาเครื่องมือเพื่อให้โมเดลทราบว่าเครื่องมือแต่ละตัวทำงานอย่างไร
- การควบคุมการดำเนินการของเครื่องมือด้วย `approval_mode`
- การส่งคืน **ผลลัพธ์ที่มีโครงสร้าง** ผ่านโมเดล Pydantic และ `response_format`

สถานการณ์คือ **เอเย่นต์การจองท่องเที่ยว** ที่สามารถค้นหาปลายทาง ตรวจสอบความพร้อม และดึงข้อมูลเที่ยวบินได้


## การตั้งค่า


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การกำหนดเครื่องมือด้วย @tool Decorator

`@tool` decorator จะเปลี่ยนฟังก์ชัน Python ธรรมดาให้กลายเป็นเครื่องมือที่เอเย่นต์สามารถเรียกใช้งานได้
จุดสำคัญ:

- **docstring** จะกลายเป็นคำอธิบายเครื่องมือที่โมเดลเห็น
- **คำอธิบายชนิดข้อมูล** (รวมถึง `Annotated` ที่มีคำอธิบาย) จะกำหนดโครงสร้างของเครื่องมือ
- `approval_mode` ควบคุมว่าผู้ใช้ต้องอนุมัติการเรียกใช้แต่ละครั้งก่อนที่จะดำเนินการหรือไม่


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## การสร้างเอเจนต์ด้วยเครื่องมือหลายตัว

ส่งเครื่องมือทั้งสามไปยังไคลเอนต์เพื่อให้โมเดลสามารถเรียกใช้งานเครื่องมือที่ต้องการเพื่อตอบคำถามของผู้ใช้ได้


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

โดยการตั้งค่า `response_format` เป็นโมเดล Pydantic ตัวเอเย่นต์จะถูกบังคับให้ส่งคืนวัตถุ JSON ที่มีชนิดข้อมูลชัดเจนแทนข้อความอิสระ ซึ่งเป็นประโยชน์เมื่อโค้ดส่วนถัดไปต้องใช้ผลลัพธ์นั้นในรูปแบบโปรแกรมเมติก


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## รูปแบบการอนุมัติเครื่องมือ

พารามิเตอร์ `approval_mode` บน `@tool` ควบคุมว่า การเรียกใช้เครื่องมือจะต้องได้รับการอนุมัติจากมนุษย์ก่อนทำการประมวลผลหรือไม่:

| โหมด | พฤติกรรม |
|---|---|
| `"never_require"` | เครื่องมือทำงานโดยอัตโนมัติ — ไม่ต้องการการยืนยันจากผู้ใช้ |
| `"always_require"` | ทุกการเรียกต้องได้รับการอนุมัติจากผู้ใช้ก่อนดำเนินการ |

ใช้ `"always_require"` สำหรับเครื่องมือที่มีผลกระทบข้างเคียง (เช่น การจองเที่ยวบิน, การชำระเงินด้วยบัตรเครดิต) เพื่อให้มีมนุษย์อยู่ในวงจรการควบคุมเสมอ


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

1. **กำหนดเครื่องมือ** โดยใช้ตัวตกแต่ง `@tool` พร้อมกับพารามิเตอร์ที่มีการระบุประเภทและ docstrings ที่ทำหน้าที่เป็นสคีมาของเครื่องมือ
2. **ผสมผสานเครื่องมือหลายตัว** เพื่อให้นายหน้าสามารถเรียกใช้พวกมันตามลำดับเพื่อตอบคำถามที่ซับซ้อน
3. **ส่งคืนผลลัพธ์ที่มีโครงสร้าง** โดยส่งโมเดล Pydantic เป็น `response_format`
4. **ควบคุมการอนุมัติเครื่องมือ** ด้วย `approval_mode` เพื่อให้มีมนุษย์เข้ามาเกี่ยวข้องในกระบวนการสำหรับการปฏิบัติที่มีความไว

รูปแบบเหล่านี้เป็นรากฐานสำหรับการสร้างเอเย่นต์ที่เชื่อถือได้ พร้อมใช้ในสภาพแวดล้อมจริงซึ่งสามารถโต้ตอบกับระบบภายนอกได้อย่างปลอดภัย


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**คำปฏิเสธความรับผิดชอบ**: เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้มีความแม่นยำ โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความผิดพลาดได้ เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาว่าเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้บริการแปลโดยมืออาชีพที่เป็นมนุษย์ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใดๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
